In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Hello world

<details>
<summary><b>Versões dos pacotes</b></summary>

O código desta página foi desenvolvido usando os seguintes requisitos.
Recomendamos usar essas versões ou mais recentes.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Este exemplo contém duas partes. Primeiro, você criará um programa quântico simples e o executará em uma unidade de processamento quântico (QPU). Como a pesquisa quântica real exige programas muito mais robustos, na segunda seção ([Escalar para grandes números de qubits](#scale-to-large-numbers-of-qubits)), você escalará o programa simples para o nível de utilidade.

## Instalar e autenticar
1. Se você ainda não instalou o Qiskit, encontre as instruções no guia de [Início rápido](/guides/quick-start).

    - Instale o Qiskit Runtime para executar jobs em hardware quântico:

        ```bash
        pip install qiskit-ibm-runtime
        ```

    - Configure um ambiente para executar notebooks Jupyter localmente:

        ```bash
        pip install jupyter
        ```

2. Configure sua autenticação para acesso ao hardware quântico através do [Open Plan](/guides/plans-overview#open-plan) gratuito.

    (Se você recebeu um convite por e-mail para ingressar em uma conta, siga as [etapas para usuários convidados](/guides/cloud-setup-invited).)

    - Acesse o [IBM Quantum Platform](https://quantum.cloud.ibm.com/) para fazer login ou criar uma conta.
         > **Note:** Se você se conectar por meio de um servidor proxy, deverá usar o Qiskit Runtime v0.44.0 ou posterior.
    - Gere sua chave de API (também chamada de *token de API*) no [painel](https://quantum.cloud.ibm.com/), e copie-a para um local seguro.
    - Acesse a página de [Instâncias](https://quantum.cloud.ibm.com/instances) e encontre a instância que deseja usar. Passe o cursor sobre o CRN e clique para copiá-lo.

    - Salve suas credenciais localmente com este código:

        ```python
        from qiskit_ibm_runtime import QiskitRuntimeService

        QiskitRuntimeService.save_account(
        token="<your-api-key>", # Use the 44-character API_KEY you created and saved from the IBM Quantum Platform Home dashboard
        instance="<CRN>", # Optional
        )
        ```

3. Agora você pode usar este código Python sempre que quiser autenticar no Qiskit Runtime Service:
    ```python
        from qiskit_ibm_runtime import QiskitRuntimeService

        # Run every time you need the service
        service = QiskitRuntimeService()
    ```
> **Info:** Se você estiver usando um computador público ou outro ambiente não seguro, siga as [instruções de autenticação manual](/guides/cloud-setup-untrusted) para manter suas credenciais de autenticação em segurança.
## Criar e executar um programa quântico simples
As quatro etapas para escrever um programa quântico usando padrões Qiskit são:

1.  Mapear o problema para um formato nativo quântico.

2.  Otimizar os circuits e operadores.

3.  Executar usando uma função primitiva quântica.

4.  Analisar os resultados.

### Etapa 1. Mapear o problema para um formato nativo quântico
Em um programa quântico, *quantum circuits* são o formato nativo para representar instruções quânticas, e *operadores* representam os observáveis a serem medidos. Ao criar um Circuit, normalmente você criará um novo objeto [`QuantumCircuit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#quantumcircuit-class) e, em seguida, adicionará instruções a ele em sequência.
A célula de código a seguir cria um Circuit que produz um *estado de Bell*, que é um estado no qual dois qubits estão totalmente entrelaçados entre si.

> **Note:** O SDK do Qiskit usa a numeração de bits LSb 0, onde o $n^{th}$ dígito tem valor $1 \ll n$ ou $2^n$. Para mais detalhes, consulte o tópico [Ordenação de bits no SDK do Qiskit](/guides/bit-ordering).

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorOptions
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from matplotlib import pyplot as plt
# Uncomment the next line if you want to use a simulator:
# from qiskit_ibm_runtime.fake_provider import FakeBelemV2


# Create a new circuit with two qubits
qc = QuantumCircuit(2)

# Add a Hadamard gate to qubit 0
qc.h(0)

# Perform a controlled-X gate on qubit 1, controlled by qubit 0
qc.cx(0, 1)

# Return a drawing of the circuit using MatPlotLib ("mpl").
# These guides are written by using Jupyter notebooks, which
# display the output of the last line of each cell.
# If you're running this in a script, use `print(qc.draw())` to
# print a text drawing.
qc.draw("mpl")

<Image src="../docs/images/guides/hello-world/extracted-outputs/930ca3b6-0.svg" alt="Output of the previous code cell" />

![Saída da célula de código anterior](../docs/images/guides/hello-world/extracted-outputs/930ca3b6-0.svg)

Consulte [`QuantumCircuit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#quantumcircuit-class) na documentação para todas as operações disponíveis.
Ao criar quantum circuits, você também deve considerar que tipo de dados deseja receber após a execução. O Qiskit oferece duas maneiras de retornar dados: você pode obter uma distribuição de probabilidade para um conjunto de qubits que escolher medir, ou pode obter o valor esperado de um observável. Prepare sua carga de trabalho para medir seu Circuit de uma dessas duas maneiras com [primitivos Qiskit](/guides/get-started-with-primitives) (explicado em detalhes na [Etapa 3](#step-3-execute-using-the-quantum-primitives)).

Este exemplo mede valores esperados usando o submódulo `qiskit.quantum_info`, que é especificado usando operadores (objetos matemáticos usados para representar uma ação ou processo que altera um estado quântico). A célula de código a seguir cria seis operadores de Pauli de dois qubits: `IZ`, `IX`, `ZI`, `XI`, `ZZ` e `XX`.

In [2]:
# Set up six different observables.

observables_labels = ["IZ", "IX", "ZI", "XI", "ZZ", "XX"]
observables = [SparsePauliOp(label) for label in observables_labels]

> **Note:** Aqui, algo como o operador `ZZ` é uma abreviação para o produto tensorial $Z\otimes Z$, o que significa medir Z no qubit 1 e Z no qubit 0 juntos, obtendo informações sobre a correlação entre o qubit 1 e o qubit 0. Valores esperados como este também são tipicamente escritos como $\langle Z_1 Z_0 \rangle$.
> 
> Se o estado estiver entrelaçado, a medição de $\langle Z_1 Z_0 \rangle$ deve ser diferente da medição de $\langle I_1 \otimes Z_0 \rangle \langle Z_1 \otimes I_0 \rangle$. Para o estado entrelaçado específico criado pelo nosso Circuit descrito acima, a medição de $\langle Z_1 Z_0 \rangle$ deve ser 1 e a medição de $\langle I_1 \otimes Z_0 \rangle \langle Z_1 \otimes I_0 \rangle$ deve ser zero.
<span id="optimize"></span>

### Etapa 2. Otimizar os circuits e operadores

Ao executar circuits em um dispositivo, é importante otimizar o conjunto de instruções que o Circuit contém e minimizar a profundidade geral (aproximadamente o número de instruções) do Circuit. Isso garante que você obtenha os melhores resultados possíveis ao reduzir os efeitos de erros e ruído. Além disso, as instruções do Circuit devem estar em conformidade com a [Arquitetura do Conjunto de Instruções (ISA)](/guides/transpile#instruction-set-architecture) de um dispositivo backend e devem considerar as gates básicas do dispositivo e a conectividade dos qubits.

O código a seguir instancia um dispositivo real para enviar um job e transforma o Circuit e os observáveis para corresponder à ISA desse backend. Requer que você já tenha [salvo suas credenciais](/guides/cloud-setup).

In [ ]:
service = QiskitRuntimeService()

backend = service.least_busy(simulator=False, operational=True)

# Convert to an ISA circuit and layout-mapped observables.
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

isa_circuit.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/hello-world/extracted-outputs/9a901271-0.svg" alt="Output of the previous code cell" />

![Saída da célula de código anterior](../docs/images/guides/hello-world/extracted-outputs/9a901271-0.svg)

### Etapa 3. Executar usando os primitivos quânticos
Os computadores quânticos podem produzir resultados aleatórios, por isso você geralmente coleta uma amostra das saídas executando o Circuit várias vezes. Você pode estimar o valor do observável usando a classe `Estimator`. O `Estimator` é um dos dois [primitivos](/guides/get-started-with-primitives); o outro é o `Sampler`, que pode ser usado para obter dados de um computador quântico. Esses objetos possuem um método `run()` que executa a seleção de circuits, observáveis e parâmetros (se aplicável), usando um [bloco unificado primitivo (PUB).](/guides/primitives#sampler)

In [4]:
# Construct the Estimator instance.

estimator = Estimator(mode=backend)
estimator.options.resilience_level = 1
estimator.options.default_shots = 5000

mapped_observables = [
    observable.apply_layout(isa_circuit.layout) for observable in observables
]

# One pub, with one circuit to run against five different observables.
job = estimator.run([(isa_circuit, mapped_observables)])

# Use the job ID to retrieve your job data later
print(f">>> Job ID: {job.job_id()}")

>>> Job ID: d5k96q4jt3vs73ds5tgg


After a job is submitted, you can wait until either the job is completed within your current python instance, or use the `job_id` to retrieve the data at a later time.  (See the [section on retrieving jobs](/docs/guides/monitor-job#retrieve-job-results-at-a-later-time) for details.)

After the job completes, examine its output through the job's `result()` attribute.

In [5]:
# This is the result of the entire submission.  You submitted one Pub,
# so this contains one inner result (and some metadata of its own).
job_result = job.result()

# This is the result from our single pub, which had six observables,
# so contains information on all six.
pub_result = job.result()[0]

In [6]:
# Check there are six observables.
# If not, edit the comments in the previous cell and update this test.
assert len(pub_result.data.evs) == 6

<Admonition type="note" title="Alternative: run the example using a simulator">
  When you run your quantum program on a real device, your workload must wait in a queue before it runs. To save time, you can instead use the following code to run this small workload on the [`fake_provider`](../api/qiskit-ibm-runtime/fake-provider) with the Qiskit Runtime local testing mode. Note that this is only possible for a small circuit. When you scale up in the next section, you will need to use a real device.

  ```python

  # Use the following code instead if you want to run on a simulator:

  from qiskit_ibm_runtime.fake_provider import FakeBelemV2
  backend = FakeBelemV2()
  estimator = Estimator(backend)

  # Convert to an ISA circuit and layout-mapped observables.

  pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
  isa_circuit = pm.run(qc)
  mapped_observables = [
      observable.apply_layout(isa_circuit.layout) for observable in observables
  ]

  job = estimator.run([(isa_circuit, mapped_observables)])
  result = job.result()

  # This is the result of the entire submission.  You submitted one Pub,
  # so this contains one inner result (and some metadata of its own).

  job_result = job.result()

  # This is the result from our single pub, which had five observables,
  # so contains information on all five.

  pub_result = job.result()[0]
  ```
</Admonition>

### Step 4. Analyze the results

The analyze step is typically where you might post-process your results using, for example, measurement error mitigation or zero noise extrapolation (ZNE). You might feed these results into another workflow for further analysis or prepare a plot of the key values and data. In general, this step is specific to your problem.  For this example, plot each of the expectation values that were measured for our circuit.

The expectation values and standard deviations for the observables you specified to Estimator are accessed through the job result's `PubResult.data.evs` and `PubResult.data.stds` attributes. To obtain the results from Sampler, use the `PubResult.data.meas.get_counts()` function, which will return a `dict` of measurements in the form of bitstrings as keys and counts as their corresponding values. For more information, see [Get started with Sampler.](/docs/guides/get-started-with-primitives#get-started-with-sampler)

In [ ]:
# Plot the result

values = pub_result.data.evs

errors = pub_result.data.stds

# plotting graph
plt.plot(observables_labels, values, "-o")
plt.xlabel("Observables")
plt.ylabel("Values")
plt.show()

<Image src="../docs/images/guides/hello-world/extracted-outputs/87143fcc-0.svg" alt="Output of the previous code cell" />

> **Note:** Quando você executa seu programa quântico em um dispositivo real, sua carga de trabalho deve aguardar na fila antes de ser executada. Para economizar tempo, você pode usar o código a seguir para executar essa pequena carga de trabalho no [`fake_provider`](../api/qiskit-ibm-runtime/fake-provider) com o modo de teste local do Qiskit Runtime. Note que isso só é possível para um Circuit pequeno. Quando você escalar na próxima seção, precisará usar um dispositivo real.
> 
>   ```python
> 
>   # Use the following code instead if you want to run on a simulator:
> 
>   from qiskit_ibm_runtime.fake_provider import FakeBelemV2
>   backend = FakeBelemV2()
>   estimator = Estimator(backend)
> 
>   # Convert to an ISA circuit and layout-mapped observables.
> 
>   pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
>   isa_circuit = pm.run(qc)
>   mapped_observables = [
>       observable.apply_layout(isa_circuit.layout) for observable in observables
>   ]
> 
>   job = estimator.run([(isa_circuit, mapped_observables)])
>   result = job.result()
> 
>   # This is the result of the entire submission.  You submitted one Pub,
>   # so this contains one inner result (and some metadata of its own).
> 
>   job_result = job.result()
> 
>   # This is the result from our single pub, which had five observables,
>   # so contains information on all five.
> 
>   pub_result = job.result()[0]
>   ```
### Etapa 4. Analisar os resultados
A etapa de análise é tipicamente onde você pode pós-processar seus resultados usando, por exemplo, mitigação de erros de medição ou extrapolação de ruído zero (ZNE). Você pode alimentar esses resultados em outro fluxo de trabalho para análise adicional ou preparar um gráfico dos valores e dados mais importantes. Em geral, essa etapa é específica para o seu problema. Para este exemplo, plote cada um dos valores esperados que foram medidos para o nosso Circuit.

Os valores esperados e desvios padrão para os observáveis que você especificou ao Estimator são acessados através dos atributos `PubResult.data.evs` e `PubResult.data.stds` do resultado do job. Para obter os resultados do Sampler, use a função `PubResult.data.meas.get_counts()`, que retornará um `dict` de medições na forma de bitstrings como chaves e contagens como seus valores correspondentes. Para mais informações, consulte [Começar com o Sampler.](/guides/get-started-with-primitives#get-started-with-sampler)

In [8]:
# Make sure the results follow the claim from the previous markdown cell.
# This can happen when the device occasionally behaves strangely. If this cell
# fails, you may just need to run the notebook again.

_results = {obs: val for obs, val in zip(observables_labels, values)}
for _label in ["IZ", "IX", "ZI", "XI"]:
    assert abs(_results[_label]) < 0.2
for _label in ["XX", "ZZ"]:
    assert _results[_label] > 0.8

![Saída da célula de código anterior](../docs/images/guides/hello-world/extracted-outputs/87143fcc-0.svg)

Observe que para os qubits 0 e 1, os valores esperados independentes de X e Z são 0, enquanto as correlações (`XX` e `ZZ`) são 1. Essa é uma marca registrada do entrelaçamento quântico.

In [ ]:
def get_qc_for_n_qubit_GHZ_state(n: int) -> QuantumCircuit:
    """This function will create a qiskit.QuantumCircuit (qc) for an n-qubit GHZ state.

    Args:
        n (int): Number of qubits in the n-qubit GHZ state

    Returns:
        QuantumCircuit: Quantum circuit that generate the n-qubit GHZ state, assuming all qubits start in the 0 state
    """
    if isinstance(n, int) and n >= 2:
        qc = QuantumCircuit(n)
        qc.h(0)
        for i in range(n - 1):
            qc.cx(i, i + 1)
    else:
        raise Exception("n is not a valid input")
    return qc


# Create a new circuit with two qubits (first argument) and two classical
# bits (second argument)
n = 100
qc = get_qc_for_n_qubit_GHZ_state(n)

## Escalar para grandes números de qubits
Na computação quântica, o trabalho em escala de utilidade é fundamental para o progresso na área. Esse trabalho requer computações em uma escala muito maior; trabalhando com circuits que podem usar mais de 100 qubits e mais de 1000 gates. Este exemplo demonstra como você pode realizar trabalho em escala de utilidade em QPUs IBM&reg; criando e analisando um estado GHZ de 100 qubits. Ele usa o fluxo de trabalho de padrões do Qiskit e termina medindo o valor esperado $\langle Z_0 Z_i \rangle $ para cada qubit.

### Etapa 1. Mapear o problema
Escreva uma função que retorna um `QuantumCircuit` que prepara um estado GHZ de $n$ qubits (essencialmente um estado de Bell estendido), e então use essa função para preparar um estado GHZ de 100 qubits e coletar os observáveis a serem medidos.

In [ ]:
# ZZII...II, ZIZI...II, ... , ZIII...IZ
operator_strings = [
    "Z" + "I" * i + "Z" + "I" * (n - 2 - i) for i in range(n - 1)
]
print(operator_strings)
print(len(operator_strings))

operators = [SparsePauliOp(operator) for operator in operator_strings]

['ZZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIIIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIIIIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIIIIIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII

Em seguida, mapeie para os operadores de interesse. Este exemplo usa os operadores `ZZ` entre qubits para examinar o comportamento à medida que eles ficam mais distantes. Valores esperados cada vez mais imprecisos (corrompidos) entre qubits distantes revelariam o nível de ruído presente.

In [ ]:
service = QiskitRuntimeService()

backend = service.least_busy(
    simulator=False, operational=True, min_num_qubits=100
)
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)

isa_circuit = pm.run(qc)
isa_operators_list = [op.apply_layout(isa_circuit.layout) for op in operators]

### Step 3. Execute on hardware

Submit the job and enable error suppression by using a technique to reduce errors called [dynamical decoupling.](../api/qiskit-ibm-runtime/options-dynamical-decoupling-options) The resilience level specifies how much resilience to build against errors. Higher levels generate more accurate results, at the expense of longer processing times.  For further explanation of the options set in the following code, see [Configure error mitigation for Qiskit Runtime.](/docs/guides/configure-error-mitigation)

In [ ]:
options = EstimatorOptions()
options.resilience_level = 1
options.dynamical_decoupling.enable = True
options.dynamical_decoupling.sequence_type = "XY4"

# Create an Estimator object
estimator = Estimator(backend, options=options)

In [13]:
# Submit the circuit to Estimator
job = estimator.run([(isa_circuit, isa_operators_list)])
job_id = job.job_id()
print(job_id)

d5k9mmqvcahs73a1ni3g


### Etapa 3. Executar no hardware
Envie o job e habilite a supressão de erros usando uma técnica para reduzir erros chamada [desacoplamento dinâmico.](../api/qiskit-ibm-runtime/options-dynamical-decoupling-options) O nível de resiliência especifica quanto resiliência construir contra erros. Níveis mais altos geram resultados mais precisos, ao custo de tempos de processamento mais longos. Para mais explicações sobre as opções definidas no código a seguir, consulte [Configurar mitigação de erros para o Qiskit Runtime.](/guides/configure-error-mitigation)

In [ ]:
# data
data = list(range(1, len(operators) + 1))  # Distance between the Z operators
result = job.result()[0]
values = result.data.evs  # Expectation value at each Z operator.
values = [
    v / values[0] for v in values
]  # Normalize the expectation values to evaluate how they decay with distance.

# plotting graph
plt.plot(data, values, marker="o", label="100-qubit GHZ state")
plt.xlabel("Distance between qubits $i$")
plt.ylabel(r"$\langle Z_i Z_0 \rangle / \langle Z_1 Z_0 \rangle $")
plt.legend()
plt.show()

<Image src="../docs/images/guides/hello-world/extracted-outputs/de91ebd0-0.svg" alt="Output of the previous code cell" />

The previous plot shows that as the distance between qubits increases, the signal decays because of the presence of noise.

## Next steps

<Admonition type="tip" title="Recommendations">
  -   Try one of these tutorials:
      - [Ground-state energy estimation of the Heisenberg chain with VQE](/docs/tutorials/spin-chain-vqe)
      - Solve optimization problems using [QAOA](/docs/tutorials/quantum-approximate-optimization-algorithm)
      - Train [quantum kernel](/docs/tutorials/quantum-kernel-training) models for machine learning tasks
  - Find detailed installation instructions in the [Install Qiskit](/docs/guides/install-qiskit) guide.
  - If you prefer not to install Qiskit locally, read about options to use Qiskit in an [online development environment.](/docs/guides/online-lab-environments)
  - To save multiple account credentials or to specify other account options, see detailed instructions in the [Save your login credentials](/docs/guides/save-credentials#save-your-access-credentials) guide.
</Admonition>